In [ ]:
import sys
sys.path.insert(0, '..')

In [ ]:
import torch
from torch import nn
from config.defaults import DefaultParams, ModelParams

In [ ]:
from src.data import test_loader, train_loader, val_loader, vocab

print(f"词表大小: {len(vocab)}")
print(f"训练 batch 数: {len(train_loader)}")
print(f"验证 batch 数: {len(val_loader)}")
print(f"测试 batch 数: {len(test_loader)}")
print(
    f"总字符数: {len(train_loader.dataset.indices) + len(val_loader.dataset.indices) + len(test_loader.dataset.indices)}"
)

In [ ]:
for i, (char, index) in enumerate(list(vocab.char2idx.items())[:15]):
    print(f"{char!r} -> {index}")
print(f"{' '.join(vocab.char2idx.keys())}")

In [ ]:
x, y = next(iter(train_loader))
print(f"x shape: {x.shape}")
print(f"y shape: {y.shape}")

print(f"Input sequence: {vocab.decode(x[0].tolist())}")
print(f"Target sequence: {vocab.decode(y[0].tolist())}")

In [ ]:
class CharRNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(len(vocab), ModelParams.EMBEDDING_DIM)
        self.rnn = nn.LSTM(
            input_size=ModelParams.EMBEDDING_DIM, # 输入维度
            hidden_size=ModelParams.HIDDEN_DIM, # 隐藏层维度
            num_layers=ModelParams.NUM_LAYERS, # 隐藏层数量
            dropout=ModelParams.DROPOUT, # dropout 概率
            batch_first=True, # 输入和输出的 batch 维度在第一维
        )
        self.fc = nn.Linear(ModelParams.HIDDEN_DIM, len(vocab)) # 输出层

    def forward(self, x, hidden=None):
        x = self.embedding(x) # (batch_size, seq_length) -> (batch_size, seq_length, embedding_dim)
        output, hidden = self.rnn(x, hidden) # (batch_size, seq_length, embedding_dim) -> (batch_size, seq_length, hidden_dim)
        output = self.fc(output) # (batch_size, seq_length, hidden_dim) -> (batch_size, seq_length, vocab_size)
        return output, hidden


In [ ]:
 
model = CharRNN().to(DefaultParams.DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f"模型总参数量: {total_params}")
print(f"词表大小: {len(vocab)}")
print(f"嵌入维度: {ModelParams.EMBEDDING_DIM}")
print(f"隐藏层维度: {ModelParams.HIDDEN_DIM}")
print(f"层数: {ModelParams.NUM_LAYERS}")
model